# DX 704 Week 6 Project

This project will develop a treatment plan for a fictious illness "Twizzleflu".
Twizzleflu is a mild illness caused by a virus.
The main symptoms are a mild fever, fidgeting, and kicking the blankets off the bed or couch.
Mild dehydration has also been reported in more severe cases.
These symptoms typically last 1-2 weeks without treatment.
Word on the internet says that Twizzleflu can be cured faster by drinking copious orange juice, but this has not been supported by evidence so far.
You will be provided with a theoretical model of Twizzleflu modeled as a Markov decision process.
Based on the model, you will compute optimal treatment plans to optimize different criteria, and compare patient discomfort with the different plans.

The full project description, a template notebook, and raw data are available on GitHub: [Project 6 Materials](https://github.com/bu-cds-dx704/dx704-project-06).

We will model Twizzleflu as a Markov decision process.
The model transition probabilities are provided in the file "twizzleflu-transitions.tsv" and the expected rewards are in "twizzleflu-rewards.tsv".
The goal for Twizzleflu is to minimize the expected discomfort of the patient which is expressed as negative rewards in the file.

## Example Code

You may find it helpful to refer to these GitHub repositories of Jupyter notebooks for example code.

* https://github.com/bu-cds-omds/dx601-examples
* https://github.com/bu-cds-omds/dx602-examples
* https://github.com/bu-cds-omds/dx603-examples
* https://github.com/bu-cds-omds/dx704-examples

Any calculations demonstrated in code examples or videos may be found in these notebooks, and you are allowed to copy this example code in your homework answers.

In [4]:
import pandas as pd
import numpy as np

In [5]:
# fns copied from dx704 examples

# copied from Implementing Value Iteration video
def compute_qT_once(R, P, gamma, v):
    return R + gamma * P @ v

# copied from Implementing Value Iteration video
def iterate_values_once(R, P, gamma, v):
    return np.max(compute_qT_once(R, P, gamma, v), axis=0)

# copied from Implementing Value Iteration video
def value_iteration(R, P, gamma, max_iterations=100, tolerance=0.001):
    # initial approximation v_0
    v_old = np.zeros(R.shape[-1])

    for i in range(max_iterations):
        # compute v_{i+1}
        v_new = iterate_values_once(R, P, gamma, v_old)

        # check if values did not change much
        if np.max(np.abs(v_new - v_old)) < tolerance:
            return v_new

        v_old = v_new

    # return v_{max_iterations}
    return v_old

# copied from Implementing Iterative Policy Evaluation Video
def iterative_policy_evaluation(R, P, gamma, pi, max_iterations=100, tolerance=0.001):
    # factor out action choices using policy.

    # deterministic version
    n = R.shape[-1]
    R_pi = R[pi, np.arange(n)]
    P_pi = P[pi, np.arange(n),:]

    # reshape to one dummy action to reuse previous example code
    R_pi = R_pi.reshape(1, *R_pi.shape)
    P_pi = P_pi.reshape(1, *P_pi.shape)

    # initial approximation v_0
    v_old = np.zeros(R.shape[-1])

    for i in range(max_iterations):
        # compute v_{i+1}
        v_new = iterate_values_once(R_pi, P_pi, gamma, v_old)

        # check if values did not change much
        if np.max(np.abs(v_new - v_old)) < tolerance:
            return v_new

        v_old = v_new

    # return v_{max_iterations}
    return v_old

def policy_iteration_baseline(R, P, gamma, max_iterations=100, tolerance=0.001):
    pi_old = np.zeros(R.shape[-1], dtype="int64")
    v_old = iterative_policy_evaluation(R, P, gamma, pi_old)

    for i in range(max_iterations):
        # compute new policy
        pi_new = np.argmax(compute_qT_once(R, P, gamma, v_old), axis=0)
        if np.array_equal(pi_new, pi_old):
            return pi_new, v_old

        # compute new values
        v_new = iterative_policy_evaluation(R, P, gamma, pi_new)
        if np.max(np.abs(v_new - v_old)) < tolerance:
            return pi_new, v_new

        pi_old = pi_new
        v_old = v_new

    return pi_old, v_old

# copied from Implementing Iterative Policy Evaluation Video
def iterative_policy_evaluation_warmstart(R, P, gamma, pi, max_iterations=100, tolerance=0.001, warmstart=None):
    # factor out action choices using policy.

    # deterministic version
    n = R.shape[-1]
    R_pi = R[pi, np.arange(n)]
    P_pi = P[pi, np.arange(n),:]

    # reshape to one dummy action to reuse previous example code
    R_pi = R_pi.reshape(1, *R_pi.shape)
    P_pi = P_pi.reshape(1, *P_pi.shape)

    # initial approximation v_0
    v_old = warmstart if warmstart is not None else np.zeros(R.shape[-1])

    for i in range(max_iterations):
        # compute v_{i+1}
        v_new = iterate_values_once(R_pi, P_pi, gamma, v_old)

        # check if values did not change much
        if np.max(np.abs(v_new - v_old)) < tolerance:
            return v_new

        v_old = v_new

    # return v_{max_iterations}
    return v_old

def policy_iteration_warmstart(R, P, gamma, max_iterations=100, tolerance=0.001):
    pi_old = np.zeros(R.shape[-1], dtype="int64")
    v_old = iterative_policy_evaluation_warmstart(R, P, gamma, pi_old)

    for i in range(max_iterations):
        # compute new policy
        pi_new = np.argmax(compute_qT_once(R, P, gamma, v_old), axis=0)
        if np.array_equal(pi_new, pi_old):
            return pi_new, v_old

        # compute new values
        v_new = iterative_policy_evaluation_warmstart(R, P, gamma, pi_new, warmstart=v_old)
        if np.max(np.abs(v_new - v_old)) < tolerance:
            return pi_new, v_new

        pi_old = pi_new
        v_old = v_new

    return pi_old, v_old

## Part 1: Evaluate a Do Nothing Plan

One of the treatment actions is to do nothing.
Calculate the expected discomfort (not rewards) of a policy that always does nothing.

Hint: for this value calculation and later ones, use value iteration.
The analytical solution has difficulties in practice when there is no discount factor.

In [6]:
rewards_df = pd.read_csv('twizzleflu-rewards.tsv', sep='\t')
rewards_df

,action,state,reward
0,do-nothing,exposed-1,0.000
1,do-nothing,exposed-2,0.000
2,do-nothing,exposed-3,0.000
3,do-nothing,symptoms-1,-0.500
4,do-nothing,symptoms-2,-1.000
5,do-nothing,symptoms-3,-0.500
6,do-nothing,recovered,0.000
7,drink-oj,exposed-1,0.000
8,drink-oj,exposed-2,0.000
9,drink-oj,exposed-3,0.000


In [7]:
transition_df = pd.read_csv('twizzleflu-transitions.tsv', sep='\t')
transition_df

,action,state,next_state,probability
0,do-nothing,exposed-1,exposed-2,0.80
1,do-nothing,exposed-1,recovered,0.20
2,do-nothing,exposed-2,exposed-3,0.80
3,do-nothing,exposed-2,recovered,0.20
4,do-nothing,exposed-3,symptoms-1,0.80
5,do-nothing,exposed-3,recovered,0.20
6,do-nothing,symptoms-1,symptoms-1,0.70
7,do-nothing,symptoms-1,symptoms-2,0.30
8,do-nothing,symptoms-2,symptoms-2,0.70
9,do-nothing,symptoms-2,symptoms-3,0.30


In [8]:
actions = rewards_df['action'].unique()
actions

<StringArray>
['do-nothing', 'drink-oj', 'sleep-8']
Length: 3, dtype: str

In [9]:
states = rewards_df['state'].unique()
states

<StringArray>
[ 'exposed-1',  'exposed-2',  'exposed-3', 'symptoms-1', 'symptoms-2',
 'symptoms-3',  'recovered']
Length: 7, dtype: str

In [10]:
# need for typing back from indices to original actions and states once we convert DFs to matrices
action_map = {action: i for i, action in enumerate(actions)}
state_map = {state: i for i, state in enumerate(states)}

In [11]:
action_map

{'do-nothing': 0, 'drink-oj': 1, 'sleep-8': 2}

In [12]:
state_map

{'exposed-1': 0,
 'exposed-2': 1,
 'exposed-3': 2,
 'symptoms-1': 3,
 'symptoms-2': 4,
 'symptoms-3': 5,
 'recovered': 6}

In [13]:
R = rewards_df.pivot_table(
    index='action', columns='state', values='reward'
).reindex(index=actions, columns=states).to_numpy()
R

array([[ 0.   ,  0.   ,  0.   , -0.5  , -1.   , -0.5  ,  0.   ],
       [ 0.   ,  0.   ,  0.   , -0.375, -0.75 , -0.375,  0.   ],
       [ 0.   ,  0.   ,  0.   , -1.   , -2.   , -1.   ,  0.   ]])

In [14]:
P = transition_df.pivot_table(
    index=['action', 'state'], columns='next_state', values='probability', fill_value=0).reindex(
        index=pd.MultiIndex.from_product([actions, states]), columns=states, fill_value=0
    ).to_numpy().reshape(len(actions), len(states), len(states))
P

array([[[0.  , 0.8 , 0.  , 0.  , 0.  , 0.  , 0.2 ],
        [0.  , 0.  , 0.8 , 0.  , 0.  , 0.  , 0.2 ],
        [0.  , 0.  , 0.  , 0.8 , 0.  , 0.  , 0.2 ],
        [0.  , 0.  , 0.  , 0.7 , 0.3 , 0.  , 0.  ],
        [0.  , 0.  , 0.  , 0.  , 0.7 , 0.3 , 0.  ],
        [0.  , 0.  , 0.  , 0.  , 0.  , 0.7 , 0.3 ],
        [0.  , 0.  , 0.  , 0.  , 0.  , 0.  , 1.  ]],

       [[0.  , 0.8 , 0.  , 0.  , 0.  , 0.  , 0.2 ],
        [0.  , 0.  , 0.8 , 0.  , 0.  , 0.  , 0.2 ],
        [0.  , 0.  , 0.  , 0.8 , 0.  , 0.  , 0.2 ],
        [0.  , 0.  , 0.  , 0.75, 0.25, 0.  , 0.  ],
        [0.  , 0.  , 0.  , 0.  , 0.75, 0.25, 0.  ],
        [0.  , 0.  , 0.  , 0.  , 0.  , 0.75, 0.25],
        [0.  , 0.  , 0.  , 0.  , 0.  , 0.  , 1.  ]],

       [[0.  , 0.5 , 0.  , 0.  , 0.  , 0.  , 0.5 ],
        [0.  , 0.  , 0.5 , 0.  , 0.  , 0.  , 0.5 ],
        [0.  , 0.  , 0.  , 0.5 , 0.  , 0.  , 0.5 ],
        [0.  , 0.  , 0.  , 0.8 , 0.2 , 0.  , 0.  ],
        [0.  , 0.  , 0.  , 0.  , 0.8 , 0.2 , 0.  ],
        

In [15]:
gamma = 1.0 # recovered is an absorbing state. basically rewards are finite so no need to discount future rewards


# this creates arr filled with index of the do-nothing action for each state
# that makes policy always do nothing
policy_do_nothing = np.full(len(states), action_map['do-nothing'], dtype=int)
policy_do_nothing

array([0, 0, 0, 0, 0, 0, 0])

In [16]:
expected_value_do_nothing = iterative_policy_evaluation(R=R, P=P, gamma=gamma, pi=policy_do_nothing)
expected_value_do_nothing

array([-3.41097781, -4.26449121, -5.33132703, -6.66481885, -4.99977911,
       -1.66665378,  0.        ])

In [17]:
do_nothing_df = pd.DataFrame({
    'state': states,
    'expected_discomfort': -expected_value_do_nothing # neg reward to pos discomfort
})
do_nothing_df

,state,expected_discomfort
0,exposed-1,3.410978
1,exposed-2,4.264491
2,exposed-3,5.331327
3,symptoms-1,6.664819
4,symptoms-2,4.999779
5,symptoms-3,1.666654
6,recovered,-0.000000


Save the expected discomfort by state to a file "do-nothing-discomfort.tsv" with columns state and expected_discomfort.

In [18]:
do_nothing_df.to_csv('do-nothing-discomfort.tsv', sep='\t', index=False)

Submit "do-nothing-discomfort.tsv" in Gradescope.

## Part 2: Compute an Optimal Treatment Plan

Compute an optimal treatment plan for Twizzleflu.
It should minimize the expected discomfort (maximize the rewards).

In [19]:
# now we want optimal policy to minimize discomfort

# i am assuming eihter warm start or the other can both be used. i use warm start here
pi_min_discomfort, v_min_discomfort = policy_iteration_warmstart(R=R, P=P, gamma=gamma)
pi_min_discomfort

array([2, 2, 2, 1, 1, 1, 0])

In [20]:
v_min_discomfort # this is the exp rewards for following the optimal policy to minimize discomfort

array([-0.75077707, -1.50123418, -3.00195708, -6.00309875, -4.50050259,
       -1.50003968,  0.        ])

Save the optimal actions for each state to a file "minimum-discomfort-actions.tsv" with columns state and action.

In [21]:
minimum_discomfort_df = pd.DataFrame({
    'state': states,
    'action': [actions[a] for a in pi_min_discomfort]
})
minimum_discomfort_df

,state,action
0,exposed-1,sleep-8
1,exposed-2,sleep-8
2,exposed-3,sleep-8
3,symptoms-1,drink-oj
4,symptoms-2,drink-oj
5,symptoms-3,drink-oj
6,recovered,do-nothing


Submit "minimum-discomfort-actions.tsv" in Gradescope.

In [22]:
minimum_discomfort_df.to_csv('minimum-discomfort-actions.tsv', sep='\t', index=False)

## Part 3: Expected Discomfort

Using your previous optimal policy, compute the expected discomfort for each state.

In [23]:
# now we have optimal policy and need expected discomfort per state
# use evalaute policy fn and look @ expected discomfort for each state. basically
# same as problem 1 except eval opt policy instead of do nothing

v_min_discomfort_per_state = iterative_policy_evaluation(R=R, P=P, gamma=gamma, pi=pi_min_discomfort)
v_min_discomfort_per_state

array([-0.74916479, -1.49868892, -2.99794399, -5.99677891, -4.49957986,
       -1.49997318,  0.        ])

In [24]:
minimum_discomfort_values = pd.DataFrame({
    'state': states,
    'expected_discomfort': -v_min_discomfort_per_state # negated to reflect discomfort
})
minimum_discomfort_values

,state,expected_discomfort
0,exposed-1,0.749165
1,exposed-2,1.498689
2,exposed-3,2.997944
3,symptoms-1,5.996779
4,symptoms-2,4.499580
5,symptoms-3,1.499973
6,recovered,-0.000000


Save your results in a file "minimum-discomfort-values.tsv" with columns state and expected_discomfort.

In [25]:
minimum_discomfort_values.to_csv('minimum-discomfort-values.tsv', sep='\t', index=False)

Submit "minimum-discomfort-values.tsv" in Gradescope.

## Part 4: Minimizing Twizzleflu Duration

Modifiy the Markov decision process to minimize the days until the Twizzle flu is over.
To do so, change the reward function to always be -1 if the current state corresponds to being sick (must have symptoms, exposed does not count) and 0 otherwise.
To be clear, the action does not matter for this reward function.


In [26]:
# now we change reward fn. current state == sick then -1, 0 otherwise
def reward_fn_new(state):
    if 'symptoms' in state:
        return -1
    return 0

In [27]:
duration_rewards = pd.DataFrame({
    'action': [a for a in actions for s in states],
    'state': [s for a in actions for s in states],
    'reward': [reward_fn_new(s) for a in actions for s in states]
})
duration_rewards

,action,state,reward
0,do-nothing,exposed-1,0
1,do-nothing,exposed-2,0
2,do-nothing,exposed-3,0
3,do-nothing,symptoms-1,-1
4,do-nothing,symptoms-2,-1
5,do-nothing,symptoms-3,-1
6,do-nothing,recovered,0
7,drink-oj,exposed-1,0
8,drink-oj,exposed-2,0
9,drink-oj,exposed-3,0


Save your new reward function in a file "duration-rewards.tsv" in the same format as "twizzleflu-rewards.tsv".

In [28]:
duration_rewards.to_csv('duration-rewards.tsv', sep='\t', index=False)

Submit "duration-rewards.tsv" in Gradescope.

## Part 5: Optimize for Shorter Twizzleflu

Compute an optimal policy to minimize the duration of Twizzleflu.

In [29]:
R_new = duration_rewards.pivot_table(
    index='action', columns='state', values='reward'
).reindex(index=actions, columns=states).to_numpy()
R_new

array([[ 0.,  0.,  0., -1., -1., -1.,  0.],
       [ 0.,  0.,  0., -1., -1., -1.,  0.],
       [ 0.,  0.,  0., -1., -1., -1.,  0.]])

In [30]:
pi_min_duration, v_min_duration = policy_iteration_warmstart(R=R_new, P=P, gamma=gamma)
pi_min_duration

array([2, 2, 2, 0, 0, 0, 0])

In [31]:
v_min_duration

array([-1.24983474, -2.49975708, -4.99964334, -9.99947692, -6.66660993,
       -3.3333303 ,  0.        ])

Save the optimal actions for each state to a file "minimum-duration-actions.tsv" with columns state and action.

In [32]:
min_duration_actions_df = pd.DataFrame({
    'state': states,
    'action': [actions[a] for a in pi_min_duration]
})
min_duration_actions_df

,state,action
0,exposed-1,sleep-8
1,exposed-2,sleep-8
2,exposed-3,sleep-8
3,symptoms-1,do-nothing
4,symptoms-2,do-nothing
5,symptoms-3,do-nothing
6,recovered,do-nothing


Submit "minimum-duration-actions.tsv" in Gradescope.

In [33]:
min_duration_actions_df.to_csv('minimum-duration-actions.tsv', sep='\t', index=False)

## Part 6: Shorter Twizzleflu?

Compute the expected number of days sick for each state to a file.

In [34]:
# repeat of prob with new reward fn
v_min_duration = iterative_policy_evaluation(R=R_new, P=P, gamma=gamma, pi=pi_min_duration)
v_min_duration

array([-1.24924413, -2.49888149, -4.9983473 , -9.99756142, -6.66636773,
       -3.33331529,  0.        ])

Save the expected sick days for each state to a file "minimum-duration-days.tsv" with columns state and expected_sick_days.

In [35]:
minimum_duration_days_df = pd.DataFrame({
    'state': states,
    'expected_sick_days': -v_min_duration
})
minimum_duration_days_df

,state,expected_sick_days
0,exposed-1,1.249244
1,exposed-2,2.498881
2,exposed-3,4.998347
3,symptoms-1,9.997561
4,symptoms-2,6.666368
5,symptoms-3,3.333315
6,recovered,-0.000000


Submit "minimum-duration-days.tsv" in Gradescope.

In [ ]:
minimum_duration_days_df.to_csv('minimum-duration-days.tsv', sep='\t', index=False)

## Part 7: Speed vs Pampering

Compute the expected discomfort using the policy to minimize days sick, and compare the results to the expected discomfort when optimizing to minimize discomfort.

Hint: Make sure that you are reporting on expected discomfort, not repeating the expected number of days.

In [37]:
# eval the actual discomfort (orig reward) under the duration minimiuzing policy
v_final = iterative_policy_evaluation(R=R, P=P, gamma=gamma, pi=pi_min_duration)
v_final

array([-0.83255615, -1.66551651, -3.33163376, -6.66415879, -4.99969234,
       -1.66664826,  0.        ])

Save the results to a file "policy-comparison.tsv" with columns state, speed_discomfort, and minimize_discomfort.

In [38]:
policy_comparison_df = pd.DataFrame({
    'state': states,
    'speed_discomfort': -v_final,
    'minimize_discomfort': -v_min_discomfort_per_state
})
policy_comparison_df

,state,speed_discomfort,minimize_discomfort
0,exposed-1,0.832556,0.749165
1,exposed-2,1.665517,1.498689
2,exposed-3,3.331634,2.997944
3,symptoms-1,6.664159,5.996779
4,symptoms-2,4.999692,4.499580
5,symptoms-3,1.666648,1.499973
6,recovered,-0.000000,-0.000000


Submit "policy-comparison.tsv" in Gradescope.

In [39]:
policy_comparison_df.to_csv('policy_comparison.tsv', sep='\t', index=False)

## Part 8: Code

Please submit a Jupyter notebook that can reproduce all your calculations and recreate the previously submitted files.

## Part 9: Acknowledgements

If you discussed this assignment with anyone, please acknowledge them here.
If you did this assignment completely on your own, simply write none below.

If you used any libraries not mentioned in this module's content, please list them with a brief explanation what you used them for. If you did not use any other libraries, simply write none below.

If you used any generative AI tools, please add links to your transcripts below, and any other information that you feel is necessary to comply with the generative AI policy. If you did not use any generative AI tools, simply write none below.